# FinSurvival Competition: Starter Notebook (Cox Model Prediction Submission)

**Objective:** This notebook provides a workflow for creating a valid prediction submission using the `CoxPHFitter` model. The competition requires you to submit a `.zip` file containing 16 separate prediction files in CSV format.

This notebook will guide you through:
1.  Loading the training and test sets for each of the 16 tasks from a single directory.
2.  Training a model (using `CoxPHFitter` as an example).
3.  Generating predictions on the test set in the required format.
4.  Saving each set of predictions to a correctly named CSV file.
5.  Zipping all 16 prediction files for submission.

## Step 1: Setup and Imports

In [ ]:
# --- import libraries ---
import pandas as pd
import numpy as np
import os
import shutil
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
from sklearn.preprocessing import StandardScaler
from typing import Tuple, Optional

## Step 2: Define a Preprocessing Function

Even though you are not submitting this code, you will still need a preprocessing pipeline to train your models effectively. You can use the one below as a starting point.

In [ ]:
##-- MAIN TO-DO -> DATA PRE-PROCESSING --##

def preprocess(train_df_with_labels: pd.DataFrame,
               test_features_df: Optional[pd.DataFrame] = None,
               ) -> Tuple[pd.DataFrame, pd.DataFrame, Optional[pd.DataFrame]]:
    """
        Preprocesses data for the competition.
    """

    # separate target variables (time, event status) and features
    train_targets = train_df_with_labels[["timeDiff", "status"]]
    train_features = train_df_with_labels.drop(columns=["timeDiff", "status"])

    # columns to drop
    cols_to_drop = ["id", "user", "pool", "Index Event", "Outcome Event", "type", "timestamp"]

    # remove unwanted columns from training features
    train_features = train_features.drop(columns=cols_to_drop, errors="ignore")

    # identify categorical columns
    categorical_cols = train_features.select_dtypes(include=["object", "category"]).columns

    # group rare categories into "Other" (keep only top 10 categories per feature)
    for col in categorical_cols:
        top_categories = train_features[col].value_counts().nlargest(10).index
        train_features[col] = train_features[col].where(train_features[col].isin(top_categories), "Other")

    # one-hot encode categorical variables
    train_features_encoded = pd.get_dummies(train_features, columns=categorical_cols, dummy_na=True, drop_first=True)

    # select numerical columns
    numerical_cols = train_features_encoded.select_dtypes(include=np.number).columns
    
    # standardize numerical features (mean=0, std=1)
    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features_encoded[numerical_cols])

    # create final processed training features
    train_features_final = pd.DataFrame(train_features_scaled, index=train_features_encoded.index, columns=numerical_cols).fillna(0)

    # drop constant features (zero variance)
    cols_to_keep = train_features_final.columns[train_features_final.var() != 0]
    train_features_final = train_features_final[cols_to_keep]


    test_processed_features = None

    if test_features_df is not None:
        # apply same preprocessing steps to test data
        test_features = test_features_df.drop(columns=cols_to_drop, errors="ignore")

        # handle categorical features in test (align with train top 10)
        for col in categorical_cols:
            top_categories = train_features[col].value_counts().nlargest(10).index
            test_features[col] = test_features[col].where(test_features[col].isin(top_categories), "Other")

        # one-hot encode test features   
        test_features_encoded = pd.get_dummies(test_features, columns=categorical_cols, dummy_na=True, drop_first=True)

        # align test columns with training columns
        train_cols = train_features_encoded.columns
        test_features_aligned = test_features_encoded.reindex(columns=train_cols, fill_value=0)

        # # scale test features using training scaler
        test_features_scaled = scaler.transform(test_features_aligned[numerical_cols])

        # create final processed test features
        test_features_final = pd.DataFrame(test_features_scaled, index=test_features_aligned.index, columns=numerical_cols).fillna(0)

        # drop constant features in test
        test_processed_features = test_features_final[cols_to_keep]

    return train_features_final, train_targets, test_processed_features

## Step 3: Loop, Train, and Save Predictions

This is the main part of the notebook. We will loop through all 16 tasks. For each task, we will:
1. Load the training data and the test features.
2. Preprocess both.
3. Train a model on the training data.
4. Generate predictions on the processed test features.
5. Save the predictions to a CSV file with the correct name.

In [ ]:
# define path to the single participant data folder.
PARTICIPANT_DATA_PATH = './participant_data/'
SUBMISSION_DIR = 'my_prediction_submission_cox'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# define all 16 event pairs
index_events = ["Borrow"]
outcome_events = index_events + ["Liquidated"]
event_pairs = []
for index_event in index_events:
    for outcome_event in outcome_events:
        if index_event == outcome_event:
            continue
        event_pairs.append((index_event, outcome_event))

for index_event, outcome_event in event_pairs:
    print(f"\n{'='*50}")
    print(f"Processing and Predicting for: {index_event} -> {outcome_event}")
    print(f"{'='*50}")
    
    dataset_path = os.path.join(index_event, outcome_event)
    
    # --- load and preprocess ---
    try:
        train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))
        # test features for participants are the validation features.
        test_features_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'test_features.csv'))
    except FileNotFoundError as e:
        print(f"Data not found for {dataset_path}. Skipping.")
        continue
        
    X_train, y_train, X_test_processed = preprocess(train_df, test_features_df)

    # --- train model ---
    try:
        lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)
        model = CoxPHFitter(penalizer=0.1)
        
        try:
            # first attempt to fit the model with standard parameters
            model.fit(lifelines_train_df, duration_col='timeDiff', event_col='status')
        except ConvergenceError:
            # if it fails, try again with a stronger penalizer and smaller step size
            print("  - Standard model failed to converge. Trying robust parameters...")
            model = CoxPHFitter(penalizer=1.0)
            model.fit(lifelines_train_df, duration_col='timeDiff', event_col='status', fit_options={'step_size': 0.1})
            print("  - Successfully converged with robust parameters.")

        # --- generate and save predictions ---
        print(f"Generating predictions for {dataset_path}...")
        predictions = -model.predict_partial_hazard(X_test_processed)
        
        # --- save predictions to a CSV file ---
        prediction_filename = dataset_path.replace(os.sep, '_') + '.csv'
        prediction_save_path = os.path.join(SUBMISSION_DIR, prediction_filename)
        pd.DataFrame(predictions).to_csv(prediction_save_path, header=False, index=False)
        print(f"  -> Predictions saved to {prediction_save_path}")
        
    except (ConvergenceError, ValueError) as e:
        print(f"\nERROR: The model for {dataset_path} failed to train even with robust parameters. No prediction file will be created.")
        print(f"Details: {e}")

print("\n\nAll prediction files have been generated.")


Processing and Predicting for: Borrow -> Liquidated
Generating predictions for Borrow/Liquidated...
  -> Predictions saved to my_prediction_submission_cox/Borrow_Liquidated.csv


All prediction files have been generated.


In [9]:
df = pd.read_csv('./my_prediction_submission_cox/Borrow_Liquidated.csv')
df.head()

,-1.2380815786284938
0,-2.540219
1,-0.693134
2,-46.301345
3,-129.796250
4,-17.425915


In [17]:
lifelines_train_df.head()

,userBorrowSum,marketWithdrawAvgAmount,marketDepositAvgAmountUSD,sinDayOfMonth,userSecondsSinceFirstTransaction,timeOfDay,userActiveDaysWeekly,userLiquidationCount,userActiveDaysMonthly,userRepayCount,...,userSecondsSincePreviousTransaction,marketBorrowSumUSD,userRepaySum,logAmountUSD,marketDepositCount,marketRepayAvgAmountUSD,sinQuarter,marketLiquidationAvgAmount,timeDiff,status
0,-0.398558,-2.489321,-2.099994,0.289315,-0.785666,-0.570639,-0.396655,-0.156334,-0.892979,-0.402879,...,-0.079284,-1.733569,-0.398471,-1.254174,-1.577749,-2.370232,1.542022,-0.932965,75264503.0,0.0
1,-0.398558,-2.487828,-2.099970,-0.306477,-0.797257,-0.545239,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.130458,-1.733569,-0.398471,-1.382937,-1.577674,-2.370232,1.542022,-0.932965,75091115.0,0.0
2,-0.398558,-2.487828,-2.097217,-0.306477,-0.797492,-0.276081,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.132896,-1.733569,-0.398471,-1.254154,-1.577637,-2.370232,1.542022,-0.932965,75084884.0,0.0
3,-0.398558,-2.487828,-2.097217,-0.306477,-0.797489,-0.274353,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.133089,-1.733569,-0.398471,-0.977241,-1.577637,-2.370232,1.542022,-0.932965,75084844.0,0.0
4,-0.398558,-2.487828,-2.095696,-0.306477,-0.797487,-0.052927,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.132848,-1.733569,-0.398471,-1.281682,-1.577600,-2.370232,1.542022,-0.932965,75079718.0,0.0


In [20]:
from lifelines.utils import concordance_index

train_risk = model.predict_partial_hazard(X_train)

# --- c-index using partial hazard as risk scores ---
c_index = concordance_index(
    lifelines_train_df["timeDiff"],       # observed times
    -train_risk,                          # negative because higher risk -> shorter survival
    lifelines_train_df["status"]          # event indicator
)
print(f"C-index: {c_index:.3f}")


C-index: 0.814
